# Phase 3 — Target Construction
**What:** create the next-hour PM2.5 target within each station.  
**Why:** turn the problem into supervised regression without cross-station leakage.  
**Output:** `data/interim/beijing_air_quality_with_target.csv`.  
**Decision:** keep only true +1-hour targets; do not impute missing targets.

## 1. Create station-safe next-hour target

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *

import pandas as pd
import numpy as np

if not CANONICAL_RAW.exists():
    raise FileNotFoundError("Run 00_dataset_integration.ipynb first.")
df=pd.read_csv(CANONICAL_RAW, parse_dates=["datetime"])
df=df.sort_values(["station","datetime"]).reset_index(drop=True)

next_time=df.groupby("station")["datetime"].shift(-1)
next_pm25=df.groupby("station")["PM2.5"].shift(-1)

df[TARGET_TIME]=next_time.where(next_time.eq(df["datetime"]+pd.Timedelta(hours=1)))
df[HAS_NEXT]=df[TARGET_TIME].notna()
df[TARGET]=next_pm25.where(df[HAS_NEXT])

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(TARGET_DATA,index=False)
print("Created:", TARGET_DATA)

Created: /Users/ruchi/Downloads/phase12_fixed/data/interim/beijing_air_quality_with_target.csv


## 2. Validate target alignment
**Decision:** every non-missing target must be exactly one hour ahead and belong to the same station.

In [2]:
valid=df[df[HAS_NEXT]]
assert valid[TARGET_TIME].sub(valid["datetime"]).eq(pd.Timedelta(hours=1)).all()
assert df.loc[~df[HAS_NEXT], TARGET_TIME].isna().all()
assert df.duplicated(["station","datetime"]).sum()==0
assert df.groupby("station")[HAS_NEXT].sum().min() > 0

target_summary={
    "rows":len(df),
    "usable_next_hour_rows":int(df[HAS_NEXT].sum()),
    "missing_target_rows":int(df[TARGET].isna().sum()),
    "target_nonnegative":bool((df[TARGET].dropna()>=0).all()),
}
display(pd.Series(target_summary).to_frame("value"))
print("Target definition: target_PM25(t) = PM2.5(t+1) at the same station.")

,value
rows,420768
usable_next_hour_rows,420756
missing_target_rows,8751
target_nonnegative,True


Target definition: target_PM25(t) = PM2.5(t+1) at the same station.


## Target-construction consequence
The last row of each station (and any row followed by a time gap) has no valid future target. Those rows stay in the dataset for audit purposes and are removed only when creating the supervised-learning dataset.